In [0]:
print("DataFrame Optimization Techniques")

In [0]:
# Read a table in samples database

cus_df = spark.table("samples.tpch.customer")
cus_df.display()

In [0]:
cus_df.rdd.getNumPartitions()

In [0]:
# 128 MB - Magic Number

In [0]:
# repartition ->  either or decrease the no of partitions

cus_df = cus_df.repartition(4, 'c_custkey')

In [0]:
cus_df.rdd.getNumPartitions()

In [0]:
cus_df.rdd.glom().collect()

In [0]:
partition_counts = cus_df.rdd.mapPartitions(lambda x: [len(list(x))]).collect()
print(partition_counts)

In [0]:
sum([187658, 187341, 187275, 187726])

In [0]:
cus_df = cus_df.repartition(10, 'c_custkey')

In [0]:
partition_counts = cus_df.rdd.mapPartitions(lambda x: [len(list(x))]).collect()
print(partition_counts)

In [0]:
# coleasce --> is just used to decrease the no.of partitions

cus_df = cus_df.coalesce(2)

In [0]:
partition_counts = cus_df.rdd.mapPartitions(lambda x: [len(list(x))]).collect()
print(partition_counts)

In [0]:
sum([374476, 375524])

In [0]:
orders_df = spark.table("samples.tpch.orders")
orders_df.display()

In [0]:
# AQE - Adaptive Query Execution
spark.conf.set("spark.databricks.optimizer.adaptive.enabled", "false")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

In [0]:
from pyspark.sql import functions as F

In [0]:
added_cus_df = (
    cus_df
    .select(
        F.col("c_custkey").alias("CustomerID"),
        F.col("c_name").alias("CustomerName")
    )
    .join(
        orders_df
        .select(
            F.col("o_custkey").alias("CustomerID"),
            F.col("o_orderkey").alias("OrderID")
        ),
        on=["CustomerID"],
        how="inner"
    )

)

added_cus_df.display()

In [0]:
# BroadCast Join

orders_df = orders_df.select(
                F.col("o_custkey").alias("CustomerID"),
                F.col("o_orderkey").alias("OrderID")
)

broadcast_cus_df = (
    cus_df
    .select(
        F.col("c_custkey").alias("CustomerID"),
        F.col("c_name").alias("CustomerName")
    )
    .join(
        F.broadcast(orders_df),
        on=["CustomerID"],
        how="inner"
    )

)

added_cus_df.display()

In [0]:
spark.conf.set("spark.databricks.optimizer.adaptive.enabled", "true")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "1048576000")

In [0]:
spark.conf.get("spark.sql.autoBroadcastJoinThreshold")

In [0]:
bit - 1
byte - 8
kb - 1024
mb - 1024 kb
gb - 1024 MB
1tb - 1024 gb

In [0]:
# BroadCast Join

# orders_df = orders_df.select(
#                 F.col("o_custkey").alias("CustomerID"),
#                 F.col("o_orderkey").alias("OrderID")
# )

cus_df = (
    cus_df
    .select(
        F.col("c_custkey").alias("CustomerID"),
        F.col("c_name").alias("CustomerName")
    )
    .join(
        orders_df.select(
            F.col("o_custkey").alias("CustomerID"),
            F.col("o_orderkey").alias("OrderID")
        ),
        on=["CustomerID"],
        how="inner"
    )

)

cus_df.show()